In [8]:
# Import required libraries

import pandas as pd
import numpy as np

print("Libraries imported successfully!")

Libraries imported successfully!


In [10]:
# Extract date-based features

df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day'] = df['date'].dt.day
df['day_of_week'] = df['date'].dt.dayofweek
df['week_of_year'] = df['date'].dt.isocalendar().week.astype(int)

# Identify weekends
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)

print("Date-based features created successfully!")

df.head()

Date-based features created successfully!


C:\Users\aditi\AppData\Local\Temp\ipykernel_19576\501252931.py:6: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  df['day_of_week'] = df['date'].dt.dayofweek


,date,store,item,sales,year,month,day,day_of_week,week_of_year,is_weekend
0,2013-01-01,1,1,13,2013,1,1,1,1,0
1,2013-01-01,1,2,33,2013,1,1,1,1,0
2,2013-01-01,1,3,15,2013,1,1,1,1,0
3,2013-01-01,1,4,10,2013,1,1,1,1,0
4,2013-01-01,1,5,11,2013,1,1,1,1,0


In [11]:
# Check the updated dataset

print("Dataset Shape:", df.shape)

print("\nUpdated Columns:")
print(df.columns.tolist())

df[['date', 'year', 'month', 'day',
    'day_of_week', 'week_of_year',
    'is_weekend', 'sales']].head(10)

Dataset Shape: (913000, 10)

Updated Columns:
['date', 'store', 'item', 'sales', 'year', 'month', 'day', 'day_of_week', 'week_of_year', 'is_weekend']


,date,year,month,day,day_of_week,week_of_year,is_weekend,sales
0,2013-01-01,2013,1,1,1,1,0,13
1,2013-01-01,2013,1,1,1,1,0,33
2,2013-01-01,2013,1,1,1,1,0,15
3,2013-01-01,2013,1,1,1,1,0,10
4,2013-01-01,2013,1,1,1,1,0,11
5,2013-01-01,2013,1,1,1,1,0,31
6,2013-01-01,2013,1,1,1,1,0,25
7,2013-01-01,2013,1,1,1,1,0,33
8,2013-01-01,2013,1,1,1,1,0,18
9,2013-01-01,2013,1,1,1,1,0,37


In [ ]:
# Update weekday feature
df['day_of_week'] = df['date'].dt.weekday

print("Weekday feature updated successfully!")

Weekday feature updated successfully!


C:\Users\aditi\AppData\Local\Temp\ipykernel_19576\1010068439.py:2: Pandas4Warning: Series.dt.weekday is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  df['day_of_week'] = df['date'].dt.weekday


In [ ]:
# Sort data for lag feature calculation

df = df.sort_values(
    by=['store', 'item', 'date']
).reset_index(drop=True)

print("Data sorted for lag feature creation!")

Data sorted for lag feature creation!


In [ ]:
# Create lag features for previous sales

df['lag_1'] = df.groupby(
    ['store', 'item']
)['sales'].shift(1)

df['lag_7'] = df.groupby(
    ['store', 'item']
)['sales'].shift(7)

print("Lag features created successfully!")

df[['date', 'store', 'item', 'sales', 'lag_1', 'lag_7']].head(12)

Lag features created successfully!


,date,store,item,sales,lag_1,lag_7
0,2013-01-01,1,1,13,NaN,NaN
1,2013-01-02,1,1,11,13.0,NaN
2,2013-01-03,1,1,14,11.0,NaN
3,2013-01-04,1,1,13,14.0,NaN
4,2013-01-05,1,1,10,13.0,NaN
5,2013-01-06,1,1,12,10.0,NaN
6,2013-01-07,1,1,10,12.0,NaN
7,2013-01-08,1,1,9,10.0,13.0
8,2013-01-09,1,1,12,9.0,11.0
9,2013-01-10,1,1,9,12.0,14.0


In [15]:
# Create 7-day rolling average feature

df['rolling_mean_7'] = df.groupby(
    ['store', 'item']
)['sales'].transform(
    lambda x: x.shift(1).rolling(window=7).mean()
)

print("7-day rolling average feature created successfully!")

df[['date', 'store', 'item', 'sales',
    'lag_1', 'lag_7', 'rolling_mean_7']].head(15)

7-day rolling average feature created successfully!


,date,store,item,sales,lag_1,lag_7,rolling_mean_7
0,2013-01-01,1,1,13,NaN,NaN,NaN
1,2013-01-02,1,1,11,13.0,NaN,NaN
2,2013-01-03,1,1,14,11.0,NaN,NaN
3,2013-01-04,1,1,13,14.0,NaN,NaN
4,2013-01-05,1,1,10,13.0,NaN,NaN
5,2013-01-06,1,1,12,10.0,NaN,NaN
6,2013-01-07,1,1,10,12.0,NaN,NaN
7,2013-01-08,1,1,9,10.0,13.0,11.857143
8,2013-01-09,1,1,12,9.0,11.0,11.285714
9,2013-01-10,1,1,9,12.0,14.0,11.428571


In [16]:
# Check missing values in newly created features

print("Missing values in engineered features:")

print(df[['lag_1', 'lag_7', 'rolling_mean_7']].isnull().sum())

Missing values in engineered features:
lag_1              500
lag_7             3500
rolling_mean_7    3500
dtype: int64


In [17]:
# Remove rows where lag and rolling features are unavailable

df = df.dropna(
    subset=['lag_1', 'lag_7', 'rolling_mean_7']
).reset_index(drop=True)

print("Missing values handled successfully!")

# Check remaining missing values
print("\nMissing values:")
print(df[['lag_1', 'lag_7', 'rolling_mean_7']].isnull().sum())

# Check updated dataset dimensions
print("\nUpdated Dataset Shape:", df.shape)

# Display the first 10 records
df[['date', 'store', 'item', 'sales',
    'lag_1', 'lag_7', 'rolling_mean_7']].head(10)

Missing values handled successfully!

Missing values:
lag_1             0
lag_7             0
rolling_mean_7    0
dtype: int64

Updated Dataset Shape: (909500, 13)


,date,store,item,sales,lag_1,lag_7,rolling_mean_7
0,2013-01-08,1,1,9,10.0,13.0,11.857143
1,2013-01-09,1,1,12,9.0,11.0,11.285714
2,2013-01-10,1,1,9,12.0,14.0,11.428571
3,2013-01-11,1,1,9,9.0,13.0,10.714286
4,2013-01-12,1,1,7,9.0,10.0,10.142857
5,2013-01-13,1,1,10,7.0,12.0,9.714286
6,2013-01-14,1,1,12,10.0,10.0,9.428571
7,2013-01-15,1,1,5,12.0,9.0,9.714286
8,2013-01-16,1,1,7,5.0,12.0,9.142857
9,2013-01-17,1,1,16,7.0,9.0,8.428571


In [18]:
# Import os for directory management
import os

# Create processed data directory if it doesn't exist
os.makedirs("../data/processed", exist_ok=True)

# Save feature-engineered dataset
df.to_csv(
    "../data/processed/feature_engineered_sales.csv",
    index=False
)

print("Feature-engineered dataset saved successfully!")
print("Location: data/processed/feature_engineered_sales.csv")

# Verify saved dataset
print("\nFinal Dataset Shape:", df.shape)

Feature-engineered dataset saved successfully!
Location: data/processed/feature_engineered_sales.csv

Final Dataset Shape: (909500, 13)
